## References
- RFC 6229: Test Vectors for the Stream Cipher RC4 (https://www.rfc-editor.org/rfc/rfc6229)
- RFC 7539: ChaCha20 and Poly1305 for IETF Protocols (https://www.rfc-editor.org/rfc/rfc7539)
- NBS Special Publication 500-20: Validating the Correctness of Hardware Implementations of the NBS Data Encryption Standard (https://www.nist.gov/publications/validating-correctness-hardware-implementations-nbs-data-encryption-standard)
- PyCryptodome Documentation: Crypto.Cipher package (ARC4, ChaCha20, ChaCha20_Poly1305, DES)
- AI Usage: Gemini and Grok (LLM) — used for verifying exact RFC test vector structures and script debugging.
- Communication with other students: No reference.

In [19]:
!pip install pycryptodome

In [20]:
from Crypto.Cipher import ARC4, ChaCha20, ChaCha20_Poly1305, DES

print("=" * 65)
print("TASK 1: VALIDATING TEST VECTORS")
print("=" * 65)

# ----------------------------------------------------------------------
# 1.1 RC4 (RFC 6229 - 40-bit key, 16-byte keystream)
# ----------------------------------------------------------------------
rc4_key = bytes.fromhex("0102030405")
expected_rc4 = bytes.fromhex("b2396305f03dc027ccc3524a0a1118a8")

rc4_cipher = ARC4.new(rc4_key)
actual_rc4 = rc4_cipher.encrypt(b"\x00" * 16)

rc4_ok = actual_rc4 == expected_rc4
print(f"[1] RC4 (RFC 6229):              {'OK (True)' if rc4_ok else 'FAIL (False)'}")
print(f"    Expected: {expected_rc4.hex()}")
print(f"    Actual:   {actual_rc4.hex()}\n")

# ----------------------------------------------------------------------
# 1.2 ChaCha20 (RFC 7539 Section 2.4.2)
# Key: 00:01:02:...:1f
# Nonce: 00:00:00:00:00:00:00:4a:00:00:00:00 (12 bytes)
# Skip 1 block (64 bytes) to emulate initial Counter=1
# ----------------------------------------------------------------------
chacha_key = bytes.fromhex("000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f")
chacha_nonce = bytes.fromhex("000000000000004a00000000")
chacha_pt = (
    b"Ladies and Gentlemen of the class of '99: If I could offer you "
    b"only one tip for the future, sunscreen would be it."
)

# Correct 114-byte Ciphertext from RFC 7539 Section 2.4.2
expected_chacha_ct = bytes.fromhex(
    "6e2e359a2568f98041ba0728dd0d6981"
    "e97e7aec1d4360c20a27afccfd9fae0b"
    "f91b65c5524733ab8f593dabcd62b357"
    "1639d624e65152ab8f530c359f0861d8"
    "07ca0dbf500d6a6156a38e088a22b65e"
    "52bc514d16ccf806818ce91ab7793736"
    "5af90bbf74a35be6b40b8eedf2785e42"
    "874d"
)

chacha_cipher = ChaCha20.new(key=chacha_key, nonce=chacha_nonce)
_ = chacha_cipher.encrypt(b"\x00" * 64)  # Skip block 0 (advance Counter to 1)
actual_chacha_ct = chacha_cipher.encrypt(chacha_pt)

chacha_ok = actual_chacha_ct == expected_chacha_ct
print(f"[2] ChaCha20 (RFC 7539):          {'OK (True)' if chacha_ok else 'FAIL (False)'}")
print(f"    Expected: {expected_chacha_ct.hex()[:40]}...")
print(f"    Actual:   {actual_chacha_ct.hex()[:40]}...\n")

# ----------------------------------------------------------------------
# 1.3 ChaCha20-Poly1305 (RFC 7539 Section 2.8.2)
# ----------------------------------------------------------------------
aead_key = bytes.fromhex("808182838485868788898a8b8c8d8e8f909192939495969798999a9b9c9d9e9f")
aead_nonce = bytes.fromhex("070000004041424344454647")
aead_aad = bytes.fromhex("50515253c0c1c2c3c4c5c6c7")
aead_pt = (
    b"Ladies and Gentlemen of the class of '99: If I could offer you "
    b"only one tip for the future, sunscreen would be it."
)
expected_aead_tag = bytes.fromhex("1ae10b594f09e26a7e902ecbd0600691")

aead_cipher = ChaCha20_Poly1305.new(key=aead_key, nonce=aead_nonce)
aead_cipher.update(aead_aad)
actual_aead_ct, actual_aead_tag = aead_cipher.encrypt_and_digest(aead_pt)

aead_ok = actual_aead_tag == expected_aead_tag
print(f"[3] ChaCha20-Poly1305 (RFC 7539): {'OK (True)' if aead_ok else 'FAIL (False)'}")
print(f"    Expected Tag: {expected_aead_tag.hex()}")
print(f"    Actual Tag:   {actual_aead_tag.hex()}\n")

# ----------------------------------------------------------------------
# 1.4 DES (NBS Special Publication 500-20, IP Test)
# ----------------------------------------------------------------------
des_key = bytes.fromhex("0101010101010101")
des_pt = bytes.fromhex("95F8A5E5DD31D900")
expected_des_ct = bytes.fromhex("8000000000000000")

des_cipher = DES.new(des_key, DES.MODE_ECB)
actual_des_ct = des_cipher.encrypt(des_pt)

des_ok = actual_des_ct == expected_des_ct
print(f"[4] DES (NBS SP 500-20):          {'OK (True)' if des_ok else 'FAIL (False)'}")
print(f"    Expected: {expected_des_ct.hex().upper()}")
print(f"    Actual:   {actual_des_ct.hex().upper()}")

TASK 1: VALIDATING TEST VECTORS
[1] RC4 (RFC 6229):              OK (True)
    Expected: b2396305f03dc027ccc3524a0a1118a8
    Actual:   b2396305f03dc027ccc3524a0a1118a8

[2] ChaCha20 (RFC 7539):          OK (True)
    Expected: 6e2e359a2568f98041ba0728dd0d6981e97e7aec...
    Actual:   6e2e359a2568f98041ba0728dd0d6981e97e7aec...

[3] ChaCha20-Poly1305 (RFC 7539): OK (True)
    Expected Tag: 1ae10b594f09e26a7e902ecbd0600691
    Actual Tag:   1ae10b594f09e26a7e902ecbd0600691

[4] DES (NBS SP 500-20):          OK (True)
    Expected: 8000000000000000
    Actual:   8000000000000000


In [21]:
def count_differing_bits(b1: bytes, b2: bytes) -> int:
    """Computes the Hamming distance (number of differing bits) between two byte strings."""
    return sum(bin(x ^ y).count("1") for x, y in zip(b1, b2))


# 128-bit (16 bytes) plaintexts
pt1 = b"0123456789abcdef"  # 16 bytes = 128 bits
# Flip exactly 1 bit (LSB of the first byte)
pt2 = bytes([pt1[0] ^ 0x01]) + pt1[1:]

print("=" * 60)
print("TASK 2: AVALANCHE EFFECT (FLIPPING 1 BIT OF PLAINTEXT)")
print("=" * 60)
print(f"Plaintext 1: {pt1} ({pt1.hex()})")
print(f"Plaintext 2: {pt2} ({pt2.hex()})")
print(
    f"Bits difference in Plaintext: {count_differing_bits(pt1, pt2)} bit\n"
)

# Common/Fixed keys and nonces for comparison
k_rc4 = b"SecretKeyForRC4!"[:16]
k_256 = b"01234567890123456789012345678901"  # 32 bytes
nonce_96 = b"nonce1234567"  # 12 bytes
k_des = b"DES_KEY!"  # 8 bytes

# --- 1. RC4 ---
c1_rc4 = ARC4.new(k_rc4).encrypt(pt1)
c2_rc4 = ARC4.new(k_rc4).encrypt(pt2)
diff_rc4 = count_differing_bits(c1_rc4, c2_rc4)

# --- 2. ChaCha20 ---
c1_chacha = ChaCha20.new(key=k_256, nonce=nonce_96).encrypt(pt1)
c2_chacha = ChaCha20.new(key=k_256, nonce=nonce_96).encrypt(pt2)
diff_chacha = count_differing_bits(c1_chacha, c2_chacha)

# --- 3. ChaCha20-Poly1305 ---
cipher_aead1 = ChaCha20_Poly1305.new(key=k_256, nonce=nonce_96)
ct1_aead, tag1 = cipher_aead1.encrypt_and_digest(pt1)

cipher_aead2 = ChaCha20_Poly1305.new(key=k_256, nonce=nonce_96)
ct2_aead, tag2 = cipher_aead2.encrypt_and_digest(pt2)

diff_aead_ct = count_differing_bits(ct1_aead, ct2_aead)
diff_aead_tag = count_differing_bits(tag1, tag2)
diff_aead_total = count_differing_bits(ct1_aead + tag1, ct2_aead + tag2)

# --- 4. DES (ECB mode, 16 bytes = 2 blocks of 8 bytes) ---
c1_des = DES.new(k_des, DES.MODE_ECB).encrypt(pt1)
c2_des = DES.new(k_des, DES.MODE_ECB).encrypt(pt2)
diff_des = count_differing_bits(c1_des, c2_des)

print(f"{'Cipher':<22} | {'Ciphertext Bits Changed':<25} | {'Notes'}")
print("-" * 75)
print(f"{'RC4':<22} | {diff_rc4:<25} | Exactly 1 bit (Stream cipher C = P ^ K)")
print(
    f"{'ChaCha20':<22} | {diff_chacha:<25} | Exactly 1 bit (Stream cipher C = P ^ K)"
)
print(
    f"{'ChaCha20-Poly1305 (CT)':<22} | {diff_aead_ct:<25} | Ciphertext payload (1 bit)"
)
print(
    f"{'ChaCha20-Poly1305 (Tag)':<22} | {diff_aead_tag:<25} | Auth Tag 128-bit MAC (~50% avalanche)"
)
print(
    f"{'ChaCha20-Poly1305 (All)':<22} | {diff_aead_total:<25} | Total message + tag changed"
)
print(
    f"{'DES (ECB Mode)':<22} | {diff_des:<25} | ~50% in Block 1 (32/64 bits), Block 2 unchanged"
)

TASK 2: AVALANCHE EFFECT (FLIPPING 1 BIT OF PLAINTEXT)
Plaintext 1: b'0123456789abcdef' (30313233343536373839616263646566)
Plaintext 2: b'1123456789abcdef' (31313233343536373839616263646566)
Bits difference in Plaintext: 1 bit

Cipher                 | Ciphertext Bits Changed   | Notes
---------------------------------------------------------------------------
RC4                    | 1                         | Exactly 1 bit (Stream cipher C = P ^ K)
ChaCha20               | 1                         | Exactly 1 bit (Stream cipher C = P ^ K)
ChaCha20-Poly1305 (CT) | 1                         | Ciphertext payload (1 bit)
ChaCha20-Poly1305 (Tag) | 66                        | Auth Tag 128-bit MAC (~50% avalanche)
ChaCha20-Poly1305 (All) | 67                        | Total message + tag changed
DES (ECB Mode)         | 41                        | ~50% in Block 1 (32/64 bits), Block 2 unchanged
